In [2]:
import pandas as pd
naturezas_path = 'naturezas.csv'
naturezas_vec = pd.read_csv(naturezas_path, sep=';')['NO_NATUREZA_INICIAL'].to_list()

emergencia = """Operador: Bom dia, precisamos verificar o que está acontecendo. Por favor, 
    descreva o que está ocorrendo.\n\nSolicitante: Deus... tem um fogo! No prédio!\n\nOperador: 
    Onde exatamente? Pode me dar o endereço?\n\nSolicitante: Praça Antônio Carlos, número seis. 
    A construção da Lorrana Negretti.\n\nOperador: Entendo. É um incêndio em um edifício 
    residencial. Você pode me dizer o que está acontecendo? As pessoas estão em perigo?\n\n
    Solicitante: Eu só cheguei há pouco tempo, vi fumaça, fogo alto... as pessoas gritando. 
    Não sei o que está pegando.\n\nOperador: Tente me dar mais detalhes. Há pessoas presas? 
    Você consegue ver o fogo de onde você está?\n\nSolicitante: Está muito forte, a fumaça 
    é muito grossa. Eu estou na praça, vendo as chamas. As pessoas estão saindo apressadas.
    \n\nOperador: Ok. Precisamos avaliar a situação. Você consegue ver se há algum ferido ou 
    pessoas em perigo imediato?\n\nSolicitante: Não consigo ver bem. A fumaça está me sufocando. 
    Apenas vejo as chamas e as pessoas correndo.\n\nOperador: Tente respirar fundo e me fale 
    se você consegue ver alguma coisa que possa ajudar a identificar a área mais afetada. 
    Há algum cheiro específico?\n\nSolicitante: Cheira muito a fumaça, queimado. Eu não sei o 
    que mais.\n\nOperador: Estamos enviando ajuda. Por favor, permaneça onde você está, a 
    menos que seja seguro fazê-lo. Você consegue me dar seu nome, por favor?\n\nSolicitante: 
    Arthur... Arthur Gabriel Porto.\n\nOperador: Ok, Arthur. Ajudamos você, Arthur. A equipe 
    de emergência já está a caminho. Fique calmo e siga as instruções da equipe quando chegarem."""


In [11]:
from typing import List
from langchain_core.runnables import chain

def test_emb_model(name: str):
    embeddings = OllamaEmbeddings(model=name)
    vector_store = InMemoryVectorStore(embeddings)
    print('Embedding with', name)
    ids = vector_store.add_documents(documents=documents)

    @chain
    def retriever(query: str) -> List[Document]:
        return vector_store.similarity_search_with_score(query, k=5)

    queries = [
        'Prédio prestes a desabar',
        'Pedro está quase se afogando na praia',
        'O prédio está pegando fogo',
        'Incêndio'
    ]

    print('Searching')

    for q, r in zip(queries, retriever.batch(queries)):
        print(q+':')
        for x in r:
            print('\t', x)

emb_models = ["nomic-embed-text", "paraphrase-multilingual", "all-minilm", "bge-m3"]
for name in emb_models:
    test_emb_model(name)

Embedding with nomic-embed-text
Searching
Prédio prestes a desabar:
	 (Document(id='bd9004e9-ffe3-4803-9927-8172589d7077', metadata={'source': 'naturezas_cad3_v2'}, page_content='Desabamento Ou Desmoronamento'), 0.7963593461715527)
	 (Document(id='73723f71-95b4-4e6c-9ccf-cf492f1c9e6e', metadata={'source': 'naturezas_cad3_v2'}, page_content='Risco de Desabamento'), 0.7853438875597101)
	 (Document(id='e91adce9-646b-476e-a20d-0ee89e9a7952', metadata={'source': 'naturezas_cad3_v2'}, page_content='Motim de Preso'), 0.7649003689958634)
	 (Document(id='42db1fac-7d9f-4f65-bd03-bc3bb0f41730', metadata={'source': 'naturezas_cad3_v2'}, page_content='Condução de Preso'), 0.7525267928066309)
	 (Document(id='37ce59a4-d23b-4516-b5e8-8a2b52430178', metadata={'source': 'naturezas_cad3_v2'}, page_content='Prevenção'), 0.7226847546371892)
Pedro está quase se afogando na praia:
	 (Document(id='8e3cc47b-5db1-4289-bccb-7424bb5bd061', metadata={'source': 'naturezas_cad3_v2'}, page_content='Objeto Preso À Pes

In [ ]:
import os
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from tqdm import tqdm

documents = [
    Document(
        page_content=natu,
        metadata={"source": "naturezas_cad3_v2"},
    )
    for natu in naturezas_vec]

embeddings = OllamaEmbeddings(model="paraphrase-multilingual")

#vectors = [embeddings.embed_query(x.page_content) for x in tqdm(documents)]
store_path = 'langchain_embeds.json'
if not os.path.exists(store_path):
    vector_store = InMemoryVectorStore(embeddings)
    ids = vector_store.add_documents(documents=documents)
    vector_store.dump(store_path)
else:
    InMemoryVectorStore.load(store_path, embeddings)


In [26]:
def search(x):
    x = vector_store.similarity_search_with_score(x, k=12)
    results = [[round(s, 4), res.page_content] for res, s in x]
    return results

In [27]:
search("Vizinho ouviu Carlos agredindo sua esposa Maria")

[[0.5625, 'Violência Contra Mulher - Maria da Penha'],
 [0.4711, 'Invasão de Domicílio'],
 [0.4426, 'Agressão Verbal'],
 [0.4305, 'Violação de Domicílio'],
 [0.4304, 'Agressão Física'],
 [0.4203, 'Pessoa Presa Em Imóvel'],
 [0.4167, 'Violência Doméstica'],
 [0.4111, 'Tentativa de Violação de Domicílio'],
 [0.4046, 'Ofensa'],
 [0.3795, 'Vítima Ejetada de Veículo'],
 [0.3774, 'Ataque de Animal'],
 [0.3747, 'Abordagem a Morador de Rua']]

In [28]:
search("Pedro teve seu carro roubado enquanto abria a garagem.")

[[0.5849, 'Roubo a Veiculo'],
 [0.5846, 'Roubo de Veículo'],
 [0.529, 'Roubo Com Tomada de Refém'],
 [0.4972, 'Remoção de Veículo Em Infração'],
 [0.4854, 'Localização de Veículo Furtado Ou Roubado'],
 [0.4843, 'Roubo À Residência'],
 [0.4819, 'Roubo a Táxi, a Transporte Alternativo Ou Por Aplicativo'],
 [0.4381, 'Roubo À Transeunte'],
 [0.4373, 'Roubo À Banco'],
 [0.4286, 'Roubo Em Transporte Público'],
 [0.4063, 'Roubo Em Transporte de Valores'],
 [0.3766, 'Furto Em Veículo']]

In [30]:
search("Patricia acourdou, foi até a cozinha, viu ela pegando fogo e saiu correndo do prédio")

[[0.4176, 'Remoção de Cadáver Vítima de Incêndio'],
 [0.4069, 'Incêndio Em Embarcação'],
 [0.3963, 'Fuga de Preso'],
 [0.3891, 'Incêndios Em Aglomerados Residenciais'],
 [0.3863, 'Incêndio Em Residência'],
 [0.3794, 'Incêndio Em Rede Elétrica'],
 [0.3752, 'Invasão de Domicílio'],
 [0.3743, 'Incêndio Em Outros'],
 [0.3594, 'Incêndio Em Lixo'],
 [0.3518, 'Incêndio Em Amontoado de Lixo/entulho'],
 [0.3515, 'Furto Em Veículo'],
 [0.35, 'Incêndio Em Vegetação']]

In [ ]:
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.prompts import PromptTemplate
import os
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from tqdm import tqdm
import json

class EmergencyInterpretationA(BaseModel):
    descricao_breve: str = Field(description="Breve descrição da ocorrência da chamada de emergência (máximo de 200 caracteres).")
    outras_observacoes: str = Field(description="Outras observações que o solicitante tenha feito durante a transcrição, mas que não estão presentes na descrição breve.")
    #tipo_chamada: str = Field(description="Tipo de ligação recebida.",
    #    enum=["Ocorrência", 'Ligação Muda', 'Trote', 'Queda de Ligação', 'Informação', 'Agradecimento', 'Denúncia'])

class NaturezaOcorrenciaA(BaseModel):
    natureza_da_ocorrencia_indice: int = Field(
        description="Indice da natureza de ocorrência mais adequada. Usar 0 (zero) quando não for nenhuma das opções.",
        default=0)

class Emergencia():
    def __init__(self, transcricao):
        self.transcricao = transcricao
        self.descricao_breve = None
        self.outras_observacoes = None
        self.classificacoes_provaveis = None
        self.classificacao_decisiva = None
    
    def to_dict(self) -> dict:
        return {
            "classificacoes_provaveis": self.classificacoes_provaveis,
            "classificacao_decisiva": self.classificacao_decisiva,
            "descricao_breve": self.descricao_breve,
            "outras_observacoes": self.outras_observacoes,
            "transcricao": self.transcricao
        }

prompt_a_template = """
You are an expert in interpreting call transcripts for emergency services. Your task is to extract relevant information
from the provided call transcript.
Please analyze the following call transcript and extract the required information:
{[[[[transcript_content]]]]}
"""

prompt_b_template = """
Você receberá a transcrição de uma chamada de emergência de ocorrência. Deve decidir qual, 
dentre uma série de naturezas de ocorrência padronizadas, é a mais adequada. Lista de naturezas:
\n{naturezas_similares}

\nTranscrição da chamada:
{[[[[transcript_content]]]]}
"""
class EmergencyInterpreterOllama():
    def __init__(self, naturezas_vec):
        self.prompt_a_template = PromptTemplate.from_template(prompt_a_template)
        self.prompt_b_template = PromptTemplate.from_template(prompt_b_template)

        self.interpret_a_structured = ChatOllama(
            model="cnmoro/gemma3-gaia-ptbr-4b:q8_0",
            temperature=0
        ).with_structured_output(
            EmergencyInterpretationA
        )

        self.interpret_b_structured = ChatOllama(
            model="cnmoro/gemma3-gaia-ptbr-4b:q8_0",
            temperature=0
        ).with_structured_output(
            NaturezaOcorrenciaA
        )

        documents = [
            Document(
                page_content=natu,
                metadata={"source": "naturezas_cad3_v2"},
            )
            for natu in naturezas_vec]

        self.embeddings = OllamaEmbeddings(model="paraphrase-multilingual")

        self.vector_store = InMemoryVectorStore(embeddings)
        self.ids = self.vector_store.add_documents(documents=documents)

    def search_naturezas(self, description, n_to_list=12):
        x = self.vector_store.similarity_search_with_score(description, k=n_to_list)
        results = [[round(s, 4), res.page_content] for res, s in x]
        return results

    def initial_interpretation(self, emergency: Emergencia):
        test_a_prompt = self.prompt_a_template.format(transcript_content=emergency.transcricao)
        result_a = self.interpret_a_structured.invoke(test_a_prompt)
        print('Result A:', result_a)
        emergency.descricao_breve = result_a.descricao_breve
        emergency.outras_observacoes = result_a.outras_observacoes
    
    def find_similar(self, emergency: Emergencia):
        classifications = search(emergency.descricao_breve)
        emergency.classificacoes_provaveis = classifications

    def decide_nature(self, emergency: Emergencia):
        classifications = emergency.classificacoes_provaveis
        classifications_indexed = {n+1: c for n, c in enumerate([x for s, x in classifications])}
        classifications_json = json.dumps(classifications_indexed, ensure_ascii=False)
        print(classifications_json)
        test_a_prompt_2 = self.prompt_b_template.format(naturezas_similares=classifications_json, 
            transcript_content=emergency.transcricao)
        natureza_correta = self.interpret_b_structured.invoke(test_a_prompt_2)
        print('natureza_correta', natureza_correta)
        nat_id = natureza_correta.natureza_da_ocorrencia_indice
        if nat_id in classifications_indexed:
            nat_name = classifications_indexed[nat_id]
        else:
            nat_name = None
        print(nat_name)
        emergency.classificacao_decisiva = nat_name

    def interpret(self, emergency: Emergencia):
        self.initial_interpretation(emergency)
        self.find_similar(emergency)
        self.decide_nature(emergency)

interpreter_instance = EmergencyInterpreterOllama(naturezas_vec)


In [19]:
emergency_instance = Emergencia(emergencia)
print(emergency_instance.to_dict())
interpreter_instance.interpret(emergency_instance)
print(json.dumps(emergency_instance.to_dict(), ensure_ascii=False, indent=2))


{'classificacoes_provaveis': None, 'classificacao_decisiva': None, 'descricao_breve': None, 'outras_observacoes': None, 'transcricao': 'Operador: Bom dia, precisamos verificar o que está acontecendo. Por favor, \n    descreva o que está ocorrendo.\n\nSolicitante: Deus... tem um fogo! No prédio!\n\nOperador: \n    Onde exatamente? Pode me dar o endereço?\n\nSolicitante: Praça Antônio Carlos, número seis. \n    A construção da Lorrana Negretti.\n\nOperador: Entendo. É um incêndio em um edifício \n    residencial. Você pode me dizer o que está acontecendo? As pessoas estão em perigo?\n\n\n    Solicitante: Eu só cheguei há pouco tempo, vi fumaça, fogo alto... as pessoas gritando. \n    Não sei o que está pegando.\n\nOperador: Tente me dar mais detalhes. Há pessoas presas? \n    Você consegue ver o fogo de onde você está?\n\nSolicitante: Está muito forte, a fumaça \n    é muito grossa. Eu estou na praça, vendo as chamas. As pessoas estão saindo apressadas.\n    \n\nOperador: Ok. Precisamos 